In [1]:
! pip install transformers --upgrade
! pip install bitsandbytes
! pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.3 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 21.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.1 MB/s eta 0:00:00:00:0100:01
  A

In [2]:
import json
import os
import csv

from torch.utils.data import Dataset


class FacebookHatefulMemeDataset(Dataset):

    def __init__(self, base_folder, split):
        self.data_set = []

        dataset_path = os.path.join(base_folder, split + '.jsonl')
        with open(dataset_path, 'r') as json_file:
            json_list = list(json_file)

        for json_str in json_list:
            result = json.loads(json_str)
            self.data_set.append(result)

    def __len__(self):
        return len(self.data_set)

    def __getitem__(self, index):
        return self.data_set[index]


class MAMIDataset(Dataset):
    def __init__(self, base_folder, split):
        self.data_set = []

        dataset_path = os.path.join(base_folder, split + '.tsv')

        # Open and parse the TSV file, directly populating self.data_set
        with open(dataset_path, 'r') as tsv_file:
            reader = csv.DictReader(tsv_file, delimiter='\t')

            for row in reader:
                data_entry = {
                    "img": "images/" + row['file_name'],
                    "label": int(row['label']),
                    "text": row['text']
                }
                self.data_set.append(data_entry)

    def __len__(self):
        return len(self.data_set)

    def __getitem__(self, index):
        return self.data_set[index]


class Harm_C_Dataset(Dataset):
    def __init__(self, base_folder, split):
        self.data_set = []

        dataset_path = os.path.join(base_folder, split + '.jsonl')
        with open(dataset_path, 'r') as json_file:
            json_list = list(json_file)

        for json_str in json_list:
            result = json.loads(json_str)
            result['label'] = 0 if 'not harmful' in result['labels'] else 1
            result['img'] = 'images/' + result['image']
            self.data_set.append(result)

    def __len__(self):
        return len(self.data_set)

    def __getitem__(self, index):
        return self.data_set[index]


class Harm_P_Dataset(Dataset):
    def __init__(self, base_folder, split):
        self.data_set = []

        dataset_path = os.path.join(base_folder, split + '.jsonl')
        with open(dataset_path, 'r') as json_file:
            json_list = list(json_file)

        for json_str in json_list:
            result = json.loads(json_str)
            result['label'] = 0 if 'not harmful' in result['labels'] else 1
            result['img'] = 'images/' + result['image']
            self.data_set.append(result)

    def __len__(self):
        return len(self.data_set)

    def __getitem__(self, index):
        return self.data_set[index]

In [3]:
from torch.utils.data import ConcatDataset


class DatasetWrapper:
    def __init__(self, dataset_tag, base_folders, split):
        self.base_folders = base_folders
        if dataset_tag == "facebook_hateful_meme_dataset":
            self.dataset = FacebookHatefulMemeDataset(
                base_folder=base_folders[0],
                split=split
            )
        elif dataset_tag == "MAMI_dataset":
            self.dataset = MAMIDataset(
                base_folder=base_folders[0],
                split=split
            )
        elif dataset_tag == "Harm_P_Dataset":
            self.dataset = Harm_P_Dataset(
                base_folder=base_folders[0],
                split=split
            )
        elif dataset_tag == "Harm_C_Dataset":
            self.dataset = Harm_C_Dataset(
                base_folder=base_folders[0],
                split=split
            )
        elif dataset_tag == "combined_Harm_CP_Dataset":
            if len(base_folders) != 2:
                raise ValueError("Both base_folder_c and base_folder_p must be provided for combined dataset.")
            self.dataset = DatasetWrapper.__get_combined_dataset(
                base_folders[0],
                base_folders[1],
                split
            )

    def get_dataset(self):
        return self.dataset

    def get_base_folders(self):
        return self.base_folders

    @staticmethod
    def __get_combined_dataset(base_folder_c, base_folder_p, split):
        covid_dataset = Harm_C_Dataset(
            base_folder_c,
            split
        )
        politics_dataset = Harm_P_Dataset(
            base_folder_p,
            split
        )
        return ConcatDataset([covid_dataset, politics_dataset])

In [4]:
# Supported Embeddings
EMBEDDINGS = [
    "clip_image",
    "blip_image",
    "sig_lip_image",
    "image_bind_image",
    "clip_image_text",
    "blip_image_text",
    "sig_lip_image_text",
    "image_bind_image_text",
    "custom"
]

# Meta Data Map
EMBEDDING_MAP = {
    # CLIP Image Encoder
    EMBEDDINGS[0]: {
        # https://codeandlife.com/2023/01/26/mastering-the-huggingface-clip-model-how-to-extract-embeddings-and-calculate-similarity-for-text-and-images/
        # https://huggingface.co/docs/transformers/en/model_doc/clip#transformers.CLIPModel.get_image_features
        "index": 0,
        "image_encoder": "openai/clip-vit-large-patch14",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/clip_image"
    },

    # BLIP Image Encoder
    EMBEDDINGS[1]: {
        # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L582
        # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L1335
        # https://huggingface.co/docs/transformers/en/model_doc/blip-2#transformers.Blip2Model.get_image_features
        "index": 1,
        "image_encoder": "Salesforce/blip2-flan-t5-xl",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/blip_image"
    },

    # SIG_LIP Image Encoder
    EMBEDDINGS[2]: {
        # https://huggingface.co/docs/transformers/en/model_doc/siglip#transformers.SiglipModel.get_image_features
        "index": 2,
        "image_encoder": "google/siglip-large-patch16-256",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/sig_lip_image"
    },

    # Image_Bind Image Encoder
    EMBEDDINGS[3]: {
        "index": 3,
        "image_encoder": "",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/image_bind_image"
    },

    # CLIP Image Text Encoder
    EMBEDDINGS[4]: {
        # https://codeandlife.com/2023/01/26/mastering-the-huggingface-clip-model-how-to-extract-embeddings-and-calculate-similarity-for-text-and-images/
        # https://huggingface.co/docs/transformers/en/model_doc/clip#transformers.CLIPModel.get_image_features
        # https://huggingface.co/docs/transformers/en/model_doc/clip#transformers.CLIPModel.get_text_features
        "index": 4,
        "image_encoder": "openai/clip-vit-large-patch14",
        "text_encoder": "openai/clip-vit-large-patch14",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/clip_image_text"
    },

    # BLIP Image Text Encoder
    EMBEDDINGS[5]: {
        # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L582
        # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L1335
        # https://huggingface.co/docs/transformers/en/model_doc/blip-2#transformers.Blip2Model.get_text_features
        # https://huggingface.co/docs/transformers/en/model_doc/blip-2#transformers.Blip2Model.get_image_features
        "index": 5,
        "image_encoder": "Salesforce/blip2-flan-t5-xl",
        "text_encoder": "Salesforce/blip2-flan-t5-xl",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/blip_image_text"
    },

    # SIG_LIP Image Text Encoder
    EMBEDDINGS[6]: {
        # https://huggingface.co/docs/transformers/en/model_doc/siglip#transformers.SiglipModel.get_text_features
        # https://huggingface.co/docs/transformers/en/model_doc/siglip#transformers.SiglipModel.get_image_features
        "index": 6,
        "image_encoder": "google/siglip-large-patch16-256",
        "text_encoder": "google/siglip-large-patch16-256",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/sig_lip_image_text"
    },

    # Image_Bind Image Text Encoder
    EMBEDDINGS[7]: {
        "index": 7,
        "image_encoder": "",
        "text_encoder": "",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/image_bind_image_text"
    },

    # Custom Embedding Generator
    EMBEDDINGS[8]: {
        "index": 8,
        "image_encoder": "",
        
        "text_encoder": "",
        "output_dir_features": "/kaggle/input/embeddings/embeddings/custom"
    },
}

In [5]:
def custom_collate_fn(batch):
    """
    Collate function for DataLoader that collates a list of dicts into a dict of lists.
    """
    collated_batch = {}
    for key in batch[0].keys():
        collated_batch[key] = [item[key] for item in batch]
    return collated_batch

In [6]:
import os

import torch
from PIL import Image
from torch.utils import data
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModel,
    Blip2Model,
    CLIPModel,
    AutoTokenizer
)



class RICES:
    def __init__(
            self,
            dataset,
            device,
            batch_size,
            embedding_details,
            base_folders,
            cached_features=None,
    ):
        self.dataset = dataset
        self.device = device
        self.batch_size = batch_size
        self.embedding_details = embedding_details
        self.base_folders = base_folders

        # Set up the model and processor
        self.__setup_model_and_processors()

        # Precompute features
        if cached_features is None:
            self.features = self.__precompute_features()
        else:
            self.features = cached_features

    def __find_image_path(self, image):
        for base_folder in self.base_folders:
            full_path = os.path.join(base_folder, image)
            if os.path.exists(full_path):
                return full_path
        raise FileNotFoundError(f"Image '{image}' not found in any of the base folders.")

    def __setup_model_and_processors(self):
        print(self.dataset, end="\n----------\n")
        print(self.embedding_details, end="\n----------\n")

        if self.embedding_details["index"] in [0, 4]:
            self.model = CLIPModel.from_pretrained(self.embedding_details["image_encoder"])
        elif self.embedding_details["index"] in [1, 5]:
            self.model = Blip2Model.from_pretrained(
                self.embedding_details["image_encoder"],
                torch_dtype=torch.bfloat16
            )
        elif self.embedding_details["index"] in [2, 6]:
            self.model = AutoModel.from_pretrained(self.embedding_details["image_encoder"])

        self.model.to(self.device)

        self.processor = AutoProcessor.from_pretrained(self.embedding_details["image_encoder"])
#         self.tokenizer = AutoTokenizer.from_pretrained(self.embedding_details["text_encoder"])

    def __model_specific_image_feature_computation(self, image_features):
        if self.embedding_details["index"] in [1, 5]:
            # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L582
            # https://github.com/huggingface/transformers/blob/main/src/transformers/models/blip_2/modeling_blip_2.py#L1335
            return image_features.pooler_output

        return image_features

    def __model_specific_text_feature_computation(self, text_features):
        if self.embedding_details["index"] in [1, 5]:
            # TODO
            pass

        return text_features

    def __precompute_features(self):
        features = []

        # Switch to evaluation mode
        self.model.eval()

        # Set up loader
        loader = torch.utils.data.DataLoader(
            self.dataset,
            batch_size=self.batch_size,
            collate_fn=custom_collate_fn,
        )

        with torch.no_grad():
            for batch in tqdm(
                    loader,
                    desc="Precomputing features for RICES",
            ):
                # Get the feature of the input image
                image_batch = batch["img"]
                print("image_batch: ")
                print(image_batch)

                inputs = self.processor(
                    images=[Image.open(self.__find_image_path(image)).convert('RGB') for image in image_batch],
                    return_tensors="pt"
                ).to(self.device)

                image_features = self.model.get_image_features(**inputs)
                image_features = self.__model_specific_image_feature_computation(image_features)

                image_features /= image_features.norm(dim=-1, keepdim=True)

                print("Image Features:", image_features.shape)

                # Get the feature of the input text
#                 text_batch = batch["text"]

#                 inputs = self.tokenizer(
#                     [text for text in text_batch],
#                     padding=True,
#                     truncation=True,
#                     return_tensors="pt"
#                 ).to(self.device)

#                 textual_features = self.model.get_text_features(**inputs)
#                 textual_features = self.__model_specific_text_feature_computation(textual_features)

#                 textual_features /= textual_features.norm(dim=-1, keepdim=True)

#                 print("Textual Features:", textual_features.shape)

                # Compute query features from image/text embeddings
                if self.embedding_details["index"] in [0, 1, 2, 3]:
                    final_features = image_features
                elif self.embedding_details["index"] in [4, 5, 6, 7]:
                    final_features = torch.cat((image_features, textual_features), -1)
                else:
                    assert self.embedding_details["index"] in [8]

                features.append(final_features.detach())
                print("Final Features:", final_features.shape, len(features))

        features = torch.cat(features)

        print(len(features), len(features[0]))
        return features

    def find(self, batch, num_examples):
        """
        Get the top num_examples most similar examples to the images.
        """
        # Switch to evaluation mode
        self.model.eval()

        with torch.no_grad():
            # Get the feature of the input image
            inputs = self.processor(
                images=[Image.open(self.__find_image_path(image)).convert('RGB') for image in batch],
                return_tensors="pt"
            ).to(self.device)

            query_feature_image = self.model.get_image_features(**inputs)
            query_feature_image = self.__model_specific_image_feature_computation(query_feature_image)

            if query_feature_image.ndim == 1:
                query_feature_image = query_feature_image.unsqueeze(0)

            query_feature_image /= query_feature_image.norm(dim=-1, keepdim=True)

#             print("Image Features:", query_feature_image.shape)

            # Get the feature of the input text
#             inputs = self.tokenizer(
#                 [image for image in batch],
#                 padding=True,
#                 truncation=True,
#                 return_tensors="pt"
#             )

#             query_feature_text = self.model.get_text_features(**inputs)
#             query_feature_text = self.__model_specific_text_feature_computation(query_feature_text)

#             if query_feature_text.ndim == 1:
#                 query_feature_text = query_feature_text.unsqueeze(0)

#             query_feature_text /= query_feature_text.norm(dim=-1, keepdim=True)

#             print("Textual Features:", query_feature_text.shape)
            
            
            
#             # Normalize both feature vectors
#             query_feature_image = (query_feature_image - query_feature_image.mean()) / query_feature_image.std()
#             query_feature_text = (query_feature_text - query_feature_text.mean()) / query_feature_text.std()
            
            
#             # Split precomputed features into image and text features
#             half_dim = self.features.shape[-1] // 2
#             precomputed_image_features = self.features[:, :half_dim]
#             precomputed_text_features = self.features[:, half_dim:]
            
#             precomputed_image_features = (precomputed_image_features - precomputed_image_features.mean()) / precomputed_image_features.std()
#             precomputed_text_features = (precomputed_text_features - precomputed_text_features.mean()) / precomputed_text_features.std()
            
#             self.features = torch.cat((precomputed_image_features, precomputed_text_features), -1)
            
            

            # Compute query features from image/text embeddings
            if self.embedding_details["index"] in [0, 1, 2, 3]:
                query_feature = query_feature_image
            elif self.embedding_details["index"] in [4, 5, 6, 7]:
                query_feature = torch.cat((query_feature_image, query_feature_text), -1)
#                 query_feature = query_feature_image + query_feature_text
            else:
                assert self.embedding_details["index"] in [8]

            query_feature = query_feature.detach().cpu()
            print("Final Features:", query_feature.shape, len(query_feature), self.features.shape)

            # Compute the similarity of the input image to the precomputed features
            similarity = (query_feature @ self.features.T).squeeze()

#             # Split precomputed features into image and text features
#             half_dim = self.features.shape[-1] // 2
#             precomputed_image_features = self.features[:, :half_dim]
#             precomputed_text_features = self.features[:, half_dim:]

#             # Perform element-wise addition of precomputed image and text features
#             precomputed_combined_features = precomputed_image_features + precomputed_text_features

#             # Compute the similarity of the input image to the precomputed features
#             similarity = (query_feature @ precomputed_combined_features.T).squeeze()

            if similarity.ndim == 1:
                similarity = similarity.unsqueeze(0)

            # Get the indices of the 'num_examples' most similar images
            indices = similarity.argsort(dim=-1, descending=True)[:, :num_examples]

        # Return with the most similar images last
        return [[self.dataset[i] for i in reversed(row)] for row in indices]

2025-04-13 16:50:28.548846: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744563028.781247      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744563028.843001      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
def load_jsonl_folder(folder_path, key="img"):
    """
    Load all JSONL files in a folder into a dictionary indexed by the specified key.
    """
    data_map = {}
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".jsonl"):
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, "r") as f:
                for line in f:
                    entry = json.loads(line)
                    data_map[entry[key]] = entry
    return data_map

def get_precomputed_data(img_path, generated_captions_map, generated_explanations_map):
    """
    Get precomputed caption and explanation for an image.
    """
    generated_captions_entry = generated_captions_map.get(img_path, {})
    generated_explanations_entry = generated_explanations_map.get(img_path, {})
    
    return {
        "caption": generated_captions_entry.get("generated_caption", ""),
        "explanation": generated_explanations_entry.get("generated_explanation", ""),
        "label": generated_explanations_entry.get("label", -1)
    }

In [8]:
# Paths to folders containing captions and explanations
generated_captions_folder = "/kaggle/input/generated-captions"
generated_explanations_folder = "/kaggle/input/generated-explanations"

# Load precomputed captions and explanations
generated_captions_map = load_jsonl_folder(generated_captions_folder, key="img")
generated_explanations_map = load_jsonl_folder(generated_explanations_folder, key="img")

In [9]:
from PIL import Image
import os


class IdeficsInference:

    def __init__(self, checkpoint):

        self.checkpoint = checkpoint

        # Hyper parameters
        self.max_new_tokens = 50

    def generate_output(
            self,
            base_folders,
            image_metadata,
            output_file,
            rices_class_few_shot,
            shots,
            dictionary,
            temperature
    ):
        selected_images = []

        if rices_class_few_shot is not None:
            selected_images = rices_class_few_shot.find([image_metadata["img"]], shots)

        prompt = self.generate_few_shot_prompts(selected_images, dictionary, base_folders)

        prompt.append(f"""\nUser: Extracted text from the image: ```{image_metadata['text']}```""")

        ### CAPTION Module in Action
        precomputed = get_precomputed_data(image_metadata["img"], generated_captions_map, generated_explanations_map)
        prompt.append(f"""\nMeme caption: ```{precomputed['caption']}```""")   # comment this if want to remove captions

        
        prompt.append(self._get_image(image_metadata["img"], base_folders))
        prompt.append("<end_of_utterance>\nAssistant:")

        inputs = self.checkpoint.processor(prompt, return_tensors="pt").to("cuda:0")

        # Generation args
        exit_condition = self.checkpoint.processor.tokenizer("<end_of_utterance>", add_special_tokens=False).input_ids
        bad_words_ids = self.checkpoint.processor.tokenizer(["<image>", "<fake_token_around_image>"],
                                                            add_special_tokens=False).input_ids

        generated_ids = self.checkpoint.model.generate(**inputs, eos_token_id=exit_condition,
                                                       bad_words_ids=bad_words_ids, max_new_tokens=self.max_new_tokens, do_sample=True, temperature= temperature)
        generated_texts = self.checkpoint.processor.batch_decode(generated_ids, skip_special_tokens=True)

        for text in generated_texts:
            print(text)
            output_file.writelines(text)

    def generate_few_shot_prompts(self, selected_images, dictionary, base_folders):
        prompt = [
            "User: Consider the following definitions.\n"
            "1. hateful - a direct or indirect attack on people based on characteristics, including ethnicity, race, "
            "nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, "
            "and disability or disease. Attack is defined as violent or dehumanizing (comparing people to non-human "
            "things, e.g. animals) speech, statements of inferiority, and calls for exclusion or segregation. Mocking "
            "hate crime is also considered hateful.\n"
            "2. not-hateful - an attack which is not hateful and follows social norms.\n"
            "Classify the meme as hateful or not-hateful based on the above definitions considering the image "
            "as well as the extracted text from the image delimited by three backticks\n"
            "Provide your answer in the format: 'hateful' or 'not-hateful', followed by a brief explanation "
            "justifying your classification.\n"
            "Example output for hateful meme : 'hateful' - Explanation within 30 words that why you classified this "
            "meme as hateful.\n"
            "Example output for not-hateful meme : 'not-hateful' - Explanation within 30 words that why you classified "
            "this meme as not-hateful."
            "<end_of_utterance>"
        ]

        for i in range(len(selected_images[0])):
            prompt.append(
                f"""\nUser: Extracted text from the image: ```{selected_images[0][i]['text']}```"""
            )
        
            # Get precomputed caption and explanation
            ### CAPTION Module in Action
            precomputed = get_precomputed_data(selected_images[0][i]["img"], generated_captions_map, generated_explanations_map)
            prompt.append(f"""\nMeme caption: ```{precomputed['caption']}```""")    # comment this if want to remove captions

            
            prompt.append(self._get_image(selected_images[0][i]["img"], base_folders))     
            prompt.append("<end_of_utterance>")
            prompt.append("\nAssistant: " + dictionary[selected_images[0][i]["label"]])

            ### EXPLANATION Module in Action
            prompt.append(f" - {precomputed['explanation']}")   # comment this if want to remove explanations
            
            prompt.append("<end_of_utterance>")

        return prompt
        
                
    def _get_image(self, image_path, base_folders):
        for base_folder in base_folders:
            full_path = os.path.join(base_folder, image_path)
            if os.path.exists(full_path):
                return Image.open(full_path)
        # If image is not found in any of the base folders, raise an error or return None as per your requirement
        raise FileNotFoundError(f"Image '{image_path}' not found in any of the base folders.")

In [10]:
from transformers import (
    IdeficsForVisionText2Text,
    AutoProcessor,
    BitsAndBytesConfig
)
import torch


class IdeficsCheckpointInitializer:

    def __init__(self, model_checkpoint):

        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,
            bnb_8bit_compute_dtype=torch.float16
        )

        # Initialize the model from model_checkpoint
        self.model = IdeficsForVisionText2Text.from_pretrained(
            model_checkpoint,
#             torch_dtype=torch.bfloat16,
            quantization_config=quantization_config,
            device_map="auto"
        )

        # Initialize the processor from model_checkpoint
        self.processor = AutoProcessor.from_pretrained(model_checkpoint)

        # Initialize inference class
        self.inference = IdeficsInference(self)

    def update_inference_class(self):
        self.inference = IdeficsInference(self)

In [11]:
import os
import os.path
import json

from tqdm import tqdm
import torch
from transformers import (
    AutoProcessor,
    AutoModel,
    Blip2Model,
    CLIPModel,
    AutoTokenizer
)


class PerformInference:
    def __init__(
            self,
            dataset_tag,
            base_folders,
            split,
            output_pickle_file_name,
            use_rices_feature,
            embed_details,
            model_checkpoint
    ):
        self.dataset_tag = dataset_tag
        self.base_folders = base_folders

        # Initialize dataset wrapper class
        self.dataset_wrapper = DatasetWrapper(
            dataset_tag=self.dataset_tag,
            base_folders=self.base_folders,
            split=split
        )

        # Initialize checkpoint initializer class
        self.checkpoint_initializer = IdeficsCheckpointInitializer(model_checkpoint)

        self.rices_few_shot = None
        if use_rices_feature:
            self.__few_shot_setup(embed_details["output_dir_features"], output_pickle_file_name, embed_details)

    def __few_shot_setup(self, output_dir_features: str, output_pickle_file_name: str, embed_details):
        # Initialize RICES class
        train_dataset_wrapper = DatasetWrapper(
            dataset_tag=self.dataset_tag,
            base_folders=self.base_folders,
            split="train"
        )

        cached_features = torch.load(
            os.path.join(output_dir_features, output_pickle_file_name),
            map_location="cpu"
        )

        self.rices_few_shot = RICES(
            dataset=train_dataset_wrapper.get_dataset(),
            device="cuda:1" if torch.cuda.is_available() else "cpu",
            batch_size=256,
            embedding_details=embed_details,
            base_folders=train_dataset_wrapper.get_base_folders(),
            cached_features=cached_features
        )

    def generate_output(
            self,
            output_directory,
            output_filename,
            labels_dictionary,
            shots,
            number_of_iterations=1,
            temperature = 0.001
    ):
        for iteration in range(number_of_iterations):
            output_file_path = os.path.join(
                output_directory,
                output_filename + "_" + str(iteration) + '.txt'
            )

            output_file = open(output_file_path, 'w')
            for image in tqdm(self.dataset_wrapper.dataset):
                output_file.writelines(json.dumps(image))
                output_file.write('\n----------\n')
                self.checkpoint_initializer.inference.generate_output(
                    self.base_folders,
                    image,
                    output_file,
                    self.rices_few_shot,
                    shots,
                    labels_dictionary,
                    temperature
                )
                output_file.write("\n##########\n")
            output_file.close()

    def dynamically_update_inference_class(self):
        self.checkpoint_initializer.update_inference_class()


embedding_details = EMBEDDING_MAP[
    EMBEDDINGS[1]
]

In [12]:
perform_inference = PerformInference(
    dataset_tag="facebook_hateful_meme_dataset",
    base_folders=["/kaggle/input/facebook-hateful-memes/archive (1)/hateful_memes/hateful_memes"],
    split="test_unseen",
    output_pickle_file_name="FHM.pkl",
    use_rices_feature=True,
    embed_details=embedding_details,
    model_checkpoint="HuggingFaceM4/idefics-9b-instruct"
)

# perform_inference = PerformInference(
#     dataset_tag="MAMI_dataset",
#     base_folders=["/kaggle/input/facebook-hateful-meme-dataset"],
#     split="test",
#     output_pickle_file_name="MAMI.pkl",
#     use_rices_feature=True,
#     embed_details=embedding_details,
#     model_checkpoint="HuggingFaceM4/idefics-9b-instruct"
# )

# perform_inference = PerformInference(
#     dataset_tag="combined_Harm_CP_Dataset",
#     base_folders=["/kaggle/input/HARM_C", "/kaggle/input/HARM_P"],
#     split="test",
#     output_pickle_file_name="HARM_CP.pkl",
#     use_rices_feature=True,
#     embed_details=embedding_details,
#     model_checkpoint="HuggingFaceM4/idefics-9b-instruct"
# )

config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/99.3k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/7.89G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

In [13]:
perform_inference.dynamically_update_inference_class()

In [22]:
perform_inference.generate_output(
    output_directory="/kaggle/working/",
    output_filename="IDEFICS_RICES_BLIP_Image_8_shots_FHM_test_unseen",
    labels_dictionary={
        0: "not-hateful",
        1: "hateful"
    },
    shots=8
)

  0%|          | 0/2000 [00:00<?, ?it/s]

Final Features: torch.Size([1, 1408]) 1 torch.Size([8500, 1408])


  0%|          | 1/2000 [00:02<1:19:14,  2.38s/it]

User: Consider the following definitions.
1. hateful - a direct or indirect attack on people based on characteristics, including ethnicity, race, nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, and disability or disease. Attack is defined as violent or dehumanizing (comparing people to non-human things, e.g. animals) speech, statements of inferiority, and calls for exclusion or segregation. Mocking hate crime is also considered hateful.
2. not-hateful - an attack which is not hateful and follows social norms.
Classify the meme as hateful or not-hateful based on the above definitions considering the image as well as the extracted text from the image delimited by three backticks
Example output for hateful meme : hateful
Example output for not hateful meme : not-hateful 
User: Extracted text from the image: ```what's the difference between boy scouts and jews? boy scouts come back from their camps``` 
Meme caption: ```Meme poster is trying to co

  0%|          | 2/2000 [00:04<1:16:46,  2.31s/it]

...

User: Consider the following definitions.
1. hateful - a direct or indirect attack on people based on characteristics, including ethnicity, race, nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, and disability or disease. Attack is defined as violent or dehumanizing (comparing people to non-human things, e.g. animals) speech, statements of inferiority, and calls for exclusion or segregation. Mocking hate crime is also considered hateful.
2. not-hateful - an attack which is not hateful and follows social norms.
Classify the meme as hateful or not-hateful based on the above definitions considering the image as well as the extracted text from the image delimited by three backticks
Example output for hateful meme : hateful
Example output for not hateful meme : not-hateful 
User: Extracted text from the image: ```thanks jews! the gift of 911``` 
Meme caption: ```Meme poster is trying to convey that the 911 attacks were a gift from the Jews.``` 
Ass

100%|█████████▉| 1999/2000 [1:11:55<00:02,  2.21s/it]

User: Consider the following definitions.
1. hateful - a direct or indirect attack on people based on characteristics, including ethnicity, race, nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, and disability or disease. Attack is defined as violent or dehumanizing (comparing people to non-human things, e.g. animals) speech, statements of inferiority, and calls for exclusion or segregation. Mocking hate crime is also considered hateful.
2. not-hateful - an attack which is not hateful and follows social norms.
Classify the meme as hateful or not-hateful based on the above definitions considering the image as well as the extracted text from the image delimited by three backticks
Example output for hateful meme : hateful
Example output for not hateful meme : not-hateful 
User: Extracted text from the image: ```what bomb``` 
Meme caption: ```Meme poster is trying to convey that the man is about to explode.``` 
Assistant: not-hateful 
User: Extrac

100%|██████████| 2000/2000 [1:11:58<00:00,  2.16s/it]

User: Consider the following definitions.
1. hateful - a direct or indirect attack on people based on characteristics, including ethnicity, race, nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, and disability or disease. Attack is defined as violent or dehumanizing (comparing people to non-human things, e.g. animals) speech, statements of inferiority, and calls for exclusion or segregation. Mocking hate crime is also considered hateful.
2. not-hateful - an attack which is not hateful and follows social norms.
Classify the meme as hateful or not-hateful based on the above definitions considering the image as well as the extracted text from the image delimited by three backticks
Example output for hateful meme : hateful
Example output for not hateful meme : not-hateful 
User: Extracted text from the image: ```you're a special kinda nuts aren't you?``` 
Meme caption: ```Meme poster is trying to convey that they are a special kind of nut.``` 
Ass

In [23]:
import json
import os.path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def parse_and_calculate_metrics(input_file_path):
    report_str=""

#     with open(input_file_path, 'r') as file:
#         lines = file.read().split("\n##########\n")
#     file.close()

#     original_json = []
#     output_text = []
#     for line in lines:
#         strings = line.split("\n----------\n")
#         if len(strings) == 2:
#             original_json.append(strings[0])
#             output_text.append(strings[1])

#     assert (len(original_json) == len(output_text))

#     original_label = []
#     predicted_label = []

#     for meme in range(len(original_json)):
#         test_label = output_text[meme].split("Assistant:")[1].strip()
#         if test_label.lower().__contains__("not-hateful") or test_label.lower().__contains__("not hateful"):
#             original_label.append(json.loads(original_json[meme])['label'])
#             predicted_label.append(0)
#         elif test_label.lower().__contains__("hateful"):
#             original_label.append(json.loads(original_json[meme])['label'])
#             predicted_label.append(1)
#         # print(meme, original_label[meme], predicted_label[meme], test_label)

#     assert (len(original_label) == len(predicted_label))

#     cnt = [0, 0, 0, 0]
#     for index in range(len(original_label)):
#         if original_label[index] == 0 and predicted_label[index] == 0:
#             cnt[0] = cnt[0] + 1
#         elif original_label[index] == 0 and predicted_label[index] == 1:
#             cnt[1] = cnt[1] + 1
#         elif original_label[index] == 1 and predicted_label[index] == 0:
#             cnt[2] = cnt[2] + 1
#         elif original_label[index] == 1 and predicted_label[index] == 1:
#             cnt[3] = cnt[3] + 1
#     report_str+= "Array for confusion matrix : OriginalLabel-PredictedLabel :\nNothateful-Nothateful, Nothateful-hateful, hateful-Nothateful, hateful-hateful\n"
#     for value in cnt:
#         report_str+= "          "
#         report_str+= str(value)
#         report_str+= "          "
#     report_str+= "\n--------------------------------------------------\n"

#     # compute the confusion matrix
#     cm = confusion_matrix(original_label, predicted_label)

#     # Plot the confusion matrix.
#     sns.heatmap(
#         cm,
#         annot=True,
#         fmt='g',
#         xticklabels=['Not hateful', 'hateful'],
#         yticklabels=['Not hateful', 'hateful']
#     )
#     plt.xlabel('Predicted Label', fontsize=10)
#     plt.ylabel('Actual Label', fontsize=10)
#     plt.title('Confusion Matrix', fontsize=20)
#     # plt.show()

#     report_str+= "Classification Report\n"
#     report_str+= classification_report(original_label, predicted_label)
#     report_str+="--------------------------------------------------\n"

#     # Finding precision and recall
#     accuracy = accuracy_score(original_label, predicted_label)
#     report_str+= "Accuracy :"
#     report_str+= str(accuracy)
#     precision = precision_score(original_label, predicted_label)
#     report_str+="\nPrecision :"
#     report_str+= str(precision)
#     recall = recall_score(original_label, predicted_label)
#     report_str+="\nRecall :"
#     report_str+= str(recall)
#     f_one_score = f1_score(original_label, predicted_label)
#     report_str+="\nF1-score :"
#     report_str+= str(f_one_score)

    return report_str

In [24]:
import email, smtplib, ssl, os

from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


def email_classification_report(folder, filename,receiver_email):
    sender_email = "testemailskgp@gmail.com"
    subject = "Classification Report of "+ filename

    # filename = "test.txt"  # In same directory as script
    filepath = os.path.join(folder, filename)

    body = parse_and_calculate_metrics(filepath)

    # password = input("Type your password and press enter:")
    password = "yqnm mjus lxil mcbp"

    # Create a multipart message and set headers
    message = MIMEMultipart()
    message["From"] = sender_email
    message["To"] = receiver_email
    message["Subject"] = subject
    message["Bcc"] = receiver_email  # Recommended for mass emails

    # Add body to email
    message.attach(MIMEText(body, "plain"))

    with open(filepath, 'rb') as attachment:
        # Add file as application/octet-stream
        # Email client can usually download this automatically as attachment
        part = MIMEBase("application", "octet-stream")
        part.set_payload(attachment.read())


    # Open PDF file in binary mode
    # with open(filename, "rb") as attachment:
    #     part = MIMEBase("application", "octet-stream")
    #     part.set_payload(attachment.read())

    # Encode file in ASCII characters to send by email    
    encoders.encode_base64(part)

    # Add header as key/value pair to attachment part
    part.add_header(
        "Content-Disposition",
        f"attachment; filename= {filename}",
    )

    # Add attachment to message and convert message to string
    message.attach(part)
    text = message.as_string()

    # Log in to server using secure context and send email
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
        server.login(sender_email, password)
        server.sendmail(sender_email, receiver_email, text)

In [25]:
receiver_email = "paramanandabhaskar@gmail.com"
folder= '/kaggle/working/'
filename = "IDEFICS_RICES_BLIP_Image_8_shots_FHM_test_unseen_0.txt"
email_classification_report(folder, filename, receiver_email)